In [1]:
import time
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, LearningRateScheduler

# Enable mixed precision for EfficientNet pipeline efficiency
tf.keras.mixed_precision.set_global_policy('mixed_float16')

# --- Configuration ---
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 25
EPOCHS = 20
INPUT_SHAPE = IMG_SIZE + (3,)

# Dummy data pipeline definition (Replace with your actual tf.data loaders)
# train_ds, val_ds, test_ds should yield (images, labels) with one-hot or sparse labels.

In [2]:
def create_vgg16_model():
    base_model = tf.keras.applications.VGG16(
        weights='imagenet', include_top=False, input_shape=INPUT_SHAPE
    )
    base_model.trainable = False  # Freeze convolutional base
    
    inputs = layers.Input(shape=INPUT_SHAPE)
    x = base_model(inputs, training=False)
    x = layers.Flatten()(x)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax', dtype='float32')(x)
    
    model = models.Model(inputs, outputs, name="VGG16_Classifier")
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

vgg16_model = create_vgg16_model()
vgg16_callbacks = [ModelCheckpoint('best_vgg16_weights.h5', save_best_only=True, monitor='val_accuracy')]
# history_vgg16 = vgg16_model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=vgg16_callbacks)

In [3]:
def create_resnet50_model():
    base_model = tf.keras.applications.ResNet50(
        weights='imagenet', include_top=False, input_shape=INPUT_SHAPE
    )
    
    # Unfreeze the last 20 layers
    base_model.trainable = True
    for layer in base_model.layers[:-20]:
        layer.trainable = False
        
    inputs = layers.Input(shape=INPUT_SHAPE)
    x = base_model(inputs, training=True)
    x = layers.GlobalAveragePooling2D()(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax', dtype='float32')(x)
    
    model = models.Model(inputs, outputs, name="ResNet50_FineTuned")
    model.compile(optimizer=optimizers.Adam(learning_rate=1e-4), loss='categorical_crossentropy', metrics=['accuracy'])
    return model

def lr_schedule(epoch):
    if epoch < 10: return 1e-4
    else: return 1e-5

resnet_model = create_resnet50_model()
resnet_callbacks = [
    ModelCheckpoint('best_resnet50_weights.h5', save_best_only=True, monitor='val_accuracy'),
    EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
    LearningRateScheduler(lr_schedule)
]
# history_resnet = resnet_model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=resnet_callbacks)

In [4]:
def create_mobilenetv2_model():
    base_model = tf.keras.applications.MobileNetV2(
        weights='imagenet', include_top=False, input_shape=INPUT_SHAPE
    )
    base_model.trainable = False
    
    # Standard light augmentation built into the model functional graph
    inputs = layers.Input(shape=INPUT_SHAPE)
    x = layers.RandomFlip("horizontal")(inputs)
    x = layers.RandomRotation(0.1)(x)
    
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax', dtype='float32')(x)
    
    model = models.Model(inputs, outputs, name="MobileNetV2_Speed")
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

mobilenet_model = create_mobilenetv2_model()
mobilenet_callbacks = [ModelCheckpoint('best_mobilenetv2_weights.h5', save_best_only=True, monitor='val_accuracy')]
# history_mobilenet = mobilenet_model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=mobilenet_callbacks)

In [5]:
def create_efficientnetb0_model():
    base_model = tf.keras.applications.EfficientNetB0(
        weights='imagenet', include_top=False, input_shape=INPUT_SHAPE
    )
    base_model.trainable = True # Set true to enable custom layer-by-layer tuning if desired
    
    inputs = layers.Input(shape=INPUT_SHAPE)
    # Advanced inline augmentations
    x = layers.RandomFlip("horizontal_and_vertical")(inputs)
    x = layers.RandomRotation(0.2)(x)
    x = layers.RandomTranslation(0.1, 0.1)(x)
    x = layers.RandomContrast(0.2)(x)
    
    x = base_model(x, training=True)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax', dtype='float32')(x)
    
    model = models.Model(inputs, outputs, name="EfficientNetB0_MaxAcc")
    model.compile(optimizer=optimizers.Adam(learning_rate=1e-4), loss='categorical_crossentropy', metrics=['accuracy'])
    return model

efficientnet_model = create_efficientnetb0_model()
efficientnet_callbacks = [ModelCheckpoint('best_efficientnetb0_weights.h5', save_best_only=True, monitor='val_accuracy')]
# history_efficientnet = efficientnet_model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=efficientnet_callbacks)

In [6]:
def evaluate_model_performance(model, x_test, y_test):
    """Calculates evaluation metrics, model size, and precise inference speeds."""
    # Compute Model Size (MB)
    model.save('temp_model.h5')
    model_size_mb = os.path.getsize('temp_model.h5') / (1024 * 1024)
    os.remove('temp_model.h5')
    
    # Calculate Latency / Inference Speed
    start_time = time.time()
    predictions = model.predict(x_test, batch_size=1)
    inference_time = (time.time() - start_time) / len(x_test)
    
    # Metrics
    y_pred_classes = np.argmax(predictions, axis=1)
    y_true_classes = np.argmax(y_test, axis=1) if len(y_test.shape) > 1 else y_test
    
    report = classification_report(y_true_classes, y_pred_classes, output_dict=True)
    
    # Top-5 Accuracy Calculation
    top_5_metric = tf.keras.metrics.TopKCategoricalAccuracy(k=5)
    top_5_metric.update_state(y_test, predictions)
    top_5_acc = top_5_metric.result().numpy()
    
    metrics = {
        'accuracy': report['accuracy'],
        'precision': report['macro avg']['precision'],
        'recall': report['macro avg']['recall'],
        'f1-score': report['macro avg']['f1-score'],
        'inference_time_per_sample': inference_time,
        'model_size_mb': model_size_mb,
        'top_5_accuracy': top_5_acc,
        'predictions': y_pred_classes,
        'raw_preds': predictions
    }
    return metrics

def plot_comparison_matrices(model_results, y_true):
    """Generates comparative plots and confusion matrices for all models."""
    fig, axes = plt.subplots(2, 2, figsize=(16, 14))
    axes = axes.ravel()
    
    for idx, (model_name, results) in enumerate(model_results.items()):
        cm = confusion_matrix(y_true, results['predictions'])
        sns.heatmap(cm, annot=False, cmap='Blues', ax=axes[idx])
        axes[idx].set_title(f"Confusion Matrix: {model_name}")
        axes[idx].set_xlabel('Predicted Label')
        axes[idx].set_ylabel('True Label')
        
    plt.tight_layout()
    plt.show()

# Example execution loop structure for step 2.5 summary:
# trained_models = {
#     "VGG16": vgg16_model, "ResNet50": resnet_model, 
#     "MobileNetV2": mobilenet_model, "EfficientNetB0": efficientnet_model
# }
# evaluation_results = {name: evaluate_model_performance(mdl, x_test, y_test) for name, mdl in trained_models.items()}
# plot_comparison_matrices(evaluation_results, np.argmax(y_test, axis=1))

In [7]:
# %% [markdown]
# # Phase 2: Transfer Learning - Image Classification Pipeline
# 
# This script contains the complete code structure for implementing the Phase 2 deep learning pipeline. 
# It sets up four target models (**VGG16, ResNet50, MobileNetV2, EfficientNetB0**) with their respective processing heads, 
# configurations, and tracking evaluations matching your specification milestones.
# 
# You can copy this script and save it as a `.py` file, or paste individual cells directly into separate Jupyter Notebook (`.ipynb`) code cells.

# %%
import time
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, LearningRateScheduler

# Enable mixed precision training globally for performance optimization (Required for Step 2.4)
tf.keras.mixed_precision.set_global_policy('mixed_float16')

# %% [markdown]
# ## 1. Pipeline Dimensions & Mock Data Structuring
# *Replace the random baseline matrices below with your actual `train_ds`, `val_ds`, and `test_ds` data feeds.*

# %%
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 25
INPUT_SHAPE = IMG_SIZE + (3,)
EPOCHS = 2  # Set to 20+ for production model converges

# Create validation tensors to ensure architectural execution limits are verified
np.random.seed(42)
x_train = np.random.rand(64, 224, 224, 3).astype(np.float32)
y_train = tf.keras.utils.to_categorical(np.random.randint(0, NUM_CLASSES, size=64), NUM_CLASSES)

x_val = np.random.rand(32, 224, 224, 3).astype(np.float32)
y_val = tf.keras.utils.to_categorical(np.random.randint(0, NUM_CLASSES, size=32), NUM_CLASSES)

x_test = np.random.rand(32, 224, 224, 3).astype(np.float32)
y_test = tf.keras.utils.to_categorical(np.random.randint(0, NUM_CLASSES, size=32), NUM_CLASSES)

# %% [markdown]
# ## Step 2.1: Model 1 - VGG16 Implementation

# %%
def build_vgg16_pipeline():
    base_model = tf.keras.applications.VGG16(weights='imagenet', include_top=False, input_shape=INPUT_SHAPE)
    base_model.trainable = False  # Completely freeze convolutional base layers
    
    inputs = layers.Input(shape=INPUT_SHAPE)
    x = base_model(inputs, training=False)
    x = layers.Flatten()(x)
    x = layers.Dense(512, activation='relu')(x)  # Custom structural dense layer
    x = layers.Dropout(0.5)(x)                   # Active dropout protection
    # Ensure final layer casts to float32 when utilizing mixed precision policies
    outputs = layers.Dense(NUM_CLASSES, activation='softmax', dtype='float32')(x)
    
    model = models.Model(inputs, outputs, name="VGG16_Classifier")
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

vgg16_model = build_vgg16_pipeline()
vgg16_callbacks = [ModelCheckpoint('best_vgg16_weights.h5', save_best_only=True, monitor='val_accuracy')]

print("--- Training VGG16 Feature Extractor ---")
history_vgg16 = vgg16_model.fit(
    x_train, y_train, 
    validation_data=(x_val, y_val), 
    epochs=EPOCHS, 
    batch_size=BATCH_SIZE, 
    callbacks=vgg16_callbacks, 
    verbose=1
)

# %% [markdown]
# ## Step 2.2: Model 2 - ResNet50 Implementation

# %%
def build_resnet50_pipeline():
    base_model = tf.keras.applications.ResNet50(weights='imagenet', include_top=False, input_shape=INPUT_SHAPE)
    
    base_model.trainable = True
    # Freeze entire base framework except for the final 20 layers
    for layer in base_model.layers[:-20]:
        layer.trainable = False
        
    inputs = layers.Input(shape=INPUT_SHAPE)
    x = base_model(inputs, training=True)
    x = layers.GlobalAveragePooling2D()(x)  # Custom Global Average Pooling
    outputs = layers.Dense(NUM_CLASSES, activation='softmax', dtype='float32')(x)
    
    model = models.Model(inputs, outputs, name="ResNet50_FineTuned")
    model.compile(optimizer=optimizers.Adam(learning_rate=1e-4), loss='categorical_crossentropy', metrics=['accuracy'])
    return model

def step_decay_schedule(epoch):
    return 1e-4 if epoch < 10 else 1e-5

resnet_model = build_resnet50_pipeline()
resnet_callbacks = [
    ModelCheckpoint('best_resnet50_weights.h5', save_best_only=True, monitor='val_accuracy'),
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    LearningRateScheduler(step_decay_schedule)
]

print("\n--- Training ResNet50 Fine-Tuning Suite ---")
history_resnet = resnet_model.fit(
    x_train, y_train, 
    validation_data=(x_val, y_val), 
    epochs=EPOCHS, 
    batch_size=BATCH_SIZE, 
    callbacks=resnet_callbacks, 
    verbose=1
)

# %% [markdown]
# ## Step 2.3: Model 3 - MobileNetV2 Implementation

# %%
def build_mobilenetv2_pipeline():
    base_model = tf.keras.applications.MobileNetV2(weights='imagenet', include_top=False, input_shape=INPUT_SHAPE)
    base_model.trainable = False  # Freeze baseline weights
    
    inputs = layers.Input(shape=INPUT_SHAPE)
    # Standard Augmentations embedded within functional model pipeline execution
    x = layers.RandomFlip("horizontal")(inputs)
    x = layers.RandomRotation(0.1)(x)
    
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax', dtype='float32')(x)
    
    model = models.Model(inputs, outputs, name="MobileNetV2_Speed")
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

mobilenet_model = build_mobilenetv2_pipeline()
mobilenet_callbacks = [ModelCheckpoint('best_mobilenetv2_weights.h5', save_best_only=True, monitor='val_accuracy')]

print("\n--- Training MobileNetV2 Inference Optimization Model ---")
history_mobilenet = mobilenet_model.fit(
    x_train, y_train, 
    validation_data=(x_val, y_val), 
    epochs=EPOCHS, 
    batch_size=BATCH_SIZE, 
    callbacks=mobilenet_callbacks, 
    verbose=1
)

# %% [markdown]
# ## Step 2.4: Model 4 - EfficientNetB0 Implementation

# %%
def build_efficientnetb0_pipeline():
    base_model = tf.keras.applications.EfficientNetB0(weights='imagenet', include_top=False, input_shape=INPUT_SHAPE)
    base_model.trainable = True  # Set open for fine tuning alignment
    
    inputs = layers.Input(shape=INPUT_SHAPE)
    # Advanced Data Augmentation Matrix
    x = layers.RandomFlip("horizontal_and_vertical")(inputs)
    x = layers.RandomRotation(0.2)(x)
    x = layers.RandomTranslation(0.1, 0.1)(x)
    x = layers.RandomContrast(0.2)(x)
    
    x = base_model(x, training=True)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)  # Head with Batch Normalization
    outputs = layers.Dense(NUM_CLASSES, activation='softmax', dtype='float32')(x)
    
    model = models.Model(inputs, outputs, name=\"EfficientNetB0_Advanced\")\n",
    model.compile(optimizer=optimizers.Adam(learning_rate=1e-4), loss='categorical_crossentropy', metrics=['accuracy'])
    return model

efficientnet_model = build_efficientnetb0_pipeline()
efficientnet_callbacks = [ModelCheckpoint('best_efficientnetb0_weights.h5', save_best_only=True, monitor='val_accuracy')]

print("\n--- Training EfficientNetB0 Compound Scaling Deep Model ---")
history_efficientnet = efficientnet_model.fit(
    x_train, y_train, 
    validation_data=(x_val, y_val), 
    epochs=EPOCHS, 
    batch_size=BATCH_SIZE, 
    callbacks=efficientnet_callbacks, 
    verbose=1
)

# %% [markdown]
# ## Step 2.5: Evaluation Matrix & Comparative Generation Engine

# %%
def evaluate_target_architecture(model, name, x_data, y_data):
    print(f"Profiling tracking logs for: {name}...")
    
    # Model Serialization Size on Disk Measurement (MB)
    temp_file = f"temp_measure_{name}.h5"
    model.save(temp_file)
    size_mb = os.path.getsize(temp_file) / (1024 * 1024)
    os.remove(temp_file)
    
    # Latency Performance Benchmark (Inference calculated sequentially over processing batch)
    start_time = time.time()
    raw_predictions = model.predict(x_data, batch_size=1, verbose=0)
    total_duration = time.time() - start_time
    latency_ms = (total_duration / len(x_data)) * 1000
    
    y_pred_classes = np.argmax(raw_predictions, axis=1)
    y_true_classes = np.argmax(y_data, axis=1)
    
    # Standard Classification Metric Maps
    report = classification_report(y_true_classes, y_pred_classes, output_dict=True, zero_division=0)
    
    # Top-5 Accuracy Verification
    top5_calculator = tf.keras.metrics.TopKCategoricalAccuracy(k=5)
    top5_calculator.update_state(y_data, raw_predictions)
    top5_acc = top5_calculator.result().numpy()
    
    return {
        'Model Name': name,
        'Accuracy (Top-1)': report['accuracy'],
        'Top-5 Accuracy': top5_acc,
        'Precision (Macro)': report['macro avg']['precision'],
        'Recall (Macro)': report['macro avg']['recall'],
        'F1-Score (Macro)': report['macro avg']['f1-score'],
        'Latency (ms/sample)': latency_ms,
        'Size (MB)': size_mb,
        'predicted_array': y_pred_classes
    }

# Execute evaluations over testing array
model_manifest = [
    (vgg16_model, "VGG16"), 
    (resnet_model, "ResNet50"), 
    (mobilenet_model, "MobileNetV2"), 
    (efficientnet_model, "EfficientNetB0")
]

evaluation_summary_records = [evaluate_target_architecture(m, name, x_test, y_test) for m, name in model_manifest]

df_metrics = pd.DataFrame(evaluation_summary_records)
print("\n=== STEP 2.5 COMPARISON METRIC RESULTS SUMMARY ===")
print(df_metrics.drop(columns=['predicted_array']))

# %% [markdown]
# ## Visualization Grid Creation

# %%
# 1. Generate Confusion Matrix Array Subplots
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.ravel()
y_true_test_labels = np.argmax(y_test, axis=1)

for idx, record in enumerate(evaluation_summary_records):
    cm = confusion_matrix(y_true_test_labels, record['predicted_array'], labels=list(range(NUM_CLASSES)))
    sns.heatmap(cm, annot=False, cmap='Blues', ax=axes[idx])
    axes[idx].set_title(f"Confusion Matrix: {record['Model Name']}")
    axes[idx].set_xlabel("Predicted Categories")
    axes[idx].set_ylabel("True Ground Labels")

plt.tight_layout()
plt.show()

# 2. Scatter Map Plot for Speed/Accuracy Efficiency Tradeoffs
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df_metrics, 
    x='Latency (ms/sample)', 
    y='Accuracy (Top-1)', 
    hue='Model Name', 
    size='Size (MB)', 
    sizes=(150, 1200), 
    alpha=0.8
)
plt.title('Accuracy vs. Inference Latency Cost Frontier (Bubble Size = Weight on Disk)')
plt.xlabel('Inference Processing Speeds (ms/sample)')
plt.ylabel('Top-1 Accuracy Evaluation Rate')
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()

SyntaxError: unexpected character after line continuation character (1532798788.py, line 176)